In [9]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_from_disk
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

# ==================== CONFIGURATION ====================
data_save_path = "/content/drive/MyDrive/CSE4078_Project/dataset_splits_hf"
output_dir = "/content/drive/MyDrive/CSE4078_Project/predictions"
num_samples = 10
BATCH_SIZE = 32

# ==================== LOAD DATASET ====================
test_dataset = load_from_disk(f"{data_save_path}/test_dataset")
test_sample = test_dataset.shuffle(seed=42).select(range(num_samples))
print(f"[INFO] Test sample ready: {len(test_sample)} examples")

# ==================== LOAD BASE MODEL ====================
model_name = "Qwen/Qwen3.5-2B"
print(f"[LOADING] Base model: {model_name}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)
model = FastLanguageModel.for_inference(model)

print("[CONFIG] Applying tokenizer adjustments...")
if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer
if hasattr(tokenizer, "processor"):
    tokenizer.processor = None
if hasattr(tokenizer, "image_processor"):
    tokenizer.image_processor = None
try:
    pure_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=True)
    tokenizer = pure_tokenizer
except Exception:
    pass
tokenizer.padding_side = "left"
model.config.use_cache = True
print("[INFO] Model and tokenizer ready")

# ==================== PREPARE PROMPTS ====================
print("[INFO] Preparing prompts...")
prompts = []
doc_titles = []
doc_specs = []
questions = []

for example in test_sample:
    doc_title = example.get("doctor_title", "Doctor")
    doc_spec = example.get("doctor_specialty") or example.get("doctor_speciality", "General Medicine")
    question = example["question_content"]

    system_prompt = (
        f"You are a Turkish medical doctor with the title '{doc_title}', "
        f"specializing in {doc_spec}. "
        "Answer patient questions in Turkish language only. "
        "Provide medically accurate, clear, professional, and empathetic responses. "
        "Base your answers strictly on medical facts. "
        "LENGTH REQUIREMENT: Keep your answer concise — maximum 5-6 sentences or ~150 words. "
        "Focus on the core medical advice: diagnosis context, recommended action, and when to seek in-person care. "
        "Avoid lengthy explanations, repetitive phrases, or listing multiple rare possibilities. "
        "Write as continuous paragraphs only: no bullet points, no headers, no markdown formatting."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    doc_titles.append(doc_title)
    doc_specs.append(doc_spec)
    questions.append(question)

# ==================== BATCHED INFERENCE ====================
results = []
total_batches = (len(prompts) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"[START] Running batched generation ({total_batches} batches of size {BATCH_SIZE})...")

for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Batching", total=total_batches):
    batch_prompts = prompts[i:i+BATCH_SIZE]

    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=128,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            use_cache=True,
        )

    input_len = inputs.input_ids.shape[1]
    for j in range(len(batch_prompts)):
        response = tokenizer.decode(outputs[j][input_len:], skip_special_tokens=True)
        response = response.replace(" <think> ", " ").replace(" </think> ", " ").strip()

        results.append({
            "doctor_title": doc_titles[i+j],
            "doctor_specialty": doc_specs[i+j],
            "question_content": questions[i+j],
            "generated_answer": response
        })

    torch.cuda.empty_cache()

# ==================== SAVE RESULTS ====================
from datetime import datetime  # Add this import if not already present

os.makedirs(output_dir, exist_ok=True)
model_slug = model_name.replace("/", "_")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(output_dir, f"{model_slug}_{num_samples}_predictions_{timestamp}.csv")
df_results = pd.DataFrame(results)
df_results.to_csv(output_path, index=False, encoding="utf-8")

print(f"[COMPLETE] Generation finished. Results saved to: {output_path}")
display(df_results.head())

[INFO] Test sample ready: 10 examples
[LOADING] Base model: Qwen/Qwen3.5-2B
==((====))==  Unsloth 2026.5.2: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[CONFIG] Applying tokenizer adjustments...
[INFO] Model and tokenizer ready
[INFO] Preparing prompts...
[START] Running batched generation (1 batches of size 32)...


Batching: 100%|██████████| 1/1 [00:10<00:00, 10.21s/it]

[COMPLETE] Generation finished. Results saved to: /content/drive/MyDrive/CSE4078_Project/predictions/Qwen_Qwen3.5-2B_10_predictions_20260509_175157.csv


,doctor_title,doctor_specialty,question_content,generated_answer
0,Doç. Dr.,ortopedi-ve-travmatoloji,Merhaba hocam sağ ayak 4.parmakta kısalık var ...,"Merhaba, Doç. Dr. olarak elimde 4. parmak uzat..."
1,Op. Dr.,beyin-ve-sinir-cerrahisi,Annem 59 yaşında ve 6 yıldır bel fıtığı rahats...,6 yıl süren bel fıtığı ve L3-S1 ileri düzeyde ...
2,Prof. Dr.,dahiliye-ve-ic-hastaliklari-nefroloji,hocam 15 aylık kızım var sol tarafta tek taraf...,"Prof. Dr. olarak, 15 aylık kızınızın sol taraf..."
3,Doç. Dr.,uroloji,"hocam merhaba, ben 45 yaşındayım. penis boyum ...","Doç. Dr. olarak, penisin içe doğru kayması ve ..."
4,Uzm. Dr.,dahiliye-ve-ic-hastaliklari,merhaba hocam helipak ilaçlarını kullanmaya ba...,Helipak (bilek) ilaçları genellikle karaciğer ...
